# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MansiNegi281/CapstoneFlyrankAI/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import pandas as pd
df = pd.read_csv("/content/content_refresh_anonymized (1).csv")
print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [3]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule :

We need to update a content item when it is not doing well. People are still searching for it. This is important for pages that are getting worse, over time are old show up low in search results and have a lot of people searching for them. If we update these pages we might get more people visiting our site again. They will be easier to find.

Reason Codes:

1.The content is not getting many visitors as it used to

2.It shows up too low in search results

3.A lot of people are searching for this content

4.The content is old and outdated

5.Not many people are clicking on it

Action Label : REFRESH_CONTENT

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

df["baseline_score"] = (
    df["search_volume"].fillna(0) * 0.30 +
    abs(df["trend_pct"].fillna(0)) * 0.40 +
    df["days_since_last_update"].fillna(0) * 0.05 +
    (40 - df["avg_position"].fillna(40)).clip(lower=0) * 2
)
def reason_code(row):

    if row["trend_direction"] == "down":
        return "DECLINING_TRAFFIC"

    elif row["avg_position"] > 20:
        return "POOR_POSITION"

    elif row["search_volume"] > 100:
        return "HIGH_VOLUME"

    elif row["days_since_last_update"] > 180:
        return "STALE_CONTENT"

    else:
        return "LOW_CTR"

df["reason_code"] = df.apply(reason_code, axis=1)

df["action_label"] = "REFRESH_CONTENT"

queue = df.sort_values(
    "baseline_score",
    ascending=False
)
queue.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action_label
12140,content_ef99c4abd9ab,client_3fdba35f04,74000.0,0.08,LOW,0.34,keyword article,informational,NaN,NaN,...,0.0,5.00,0.0,good,page_3_5,stable,3.6,22209.64,POOR_POSITION,REFRESH_CONTENT
17907,content_5ec29ae79c60,client_3fdba35f04,60500.0,0.13,LOW,0.76,keyword article,informational,NaN,NaN,...,0.0,0.00,0.0,moderate,page_3_5,up,73.0,18184.40,POOR_POSITION,REFRESH_CONTENT
28282,content_454cc6654c6e,client_3fdba35f04,60500.0,0.11,LOW,0.39,keyword article,informational,NaN,NaN,...,0.0,4.76,0.0,good,page_3_5,down,-52.8,18176.32,DECLINING_TRAFFIC,REFRESH_CONTENT
6972,content_bf67a444faef,client_3fdba35f04,60500.0,0.11,LOW,0.50,keyword article,informational,NaN,NaN,...,0.0,13.33,0.0,good,page_3_5,down,-20.0,18163.20,DECLINING_TRAFFIC,REFRESH_CONTENT
18701,content_deb54e9e19cd,client_3fdba35f04,60500.0,0.13,LOW,0.56,keyword article,informational,NaN,NaN,...,0.0,0.00,0.0,good,page_3_5,stable,-2.5,18156.20,POOR_POSITION,REFRESH_CONTENT


In [5]:
import os
os.makedirs("work/outputs", exist_ok=True)
queue[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label"
    ]
].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)
print("CSV Saved")

CSV Saved


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

1. content_ef99c4abd9ab was prioritized due to high search volume and poor ranking. This could be wrong if competition is the main issue.
2. content_5ec29ae79c60 has strong search demand but ranks poorly, making it a good optimization candidate. This may be wrong if search intent is already well matched.
3. content_454cc6654c6e was selected because traffic is declining significantly. This could be due to seasonality rather than content quality.
4. content_bf67a444faef shows declining traffic and may benefit from a refresh. The decline could be temporary.
5. content_deb54e9e19cd has high volume but low rankings. This may be caused by technical or competitive factors.
6. content_dd882c4152ac ranks poorly but has low search volume, making it a weaker recommendation. The score may be influenced by unusual trend values.
7. content_83e3da1394ac combines high demand with poor rankings, suggesting optimization potential. Competition may limit improvement.
8. content_f76ccf7a7834 has declining traffic despite a decent position. Reduced search demand may be the real cause.
9. content_ee4630879d03 was selected due to traffic decline and moderate rankings. Seasonal effects could explain the drop.
10. content_cd6760921db8 experienced a large traffic decline and appears worth reviewing. External factors may be responsible.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20)
top20[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "search_volume",
        "avg_position",
        "trend_direction",
        "trend_pct"
    ]
]


,content_id,baseline_score,reason_code,search_volume,avg_position,trend_direction,trend_pct
12140,content_ef99c4abd9ab,22209.64,POOR_POSITION,74000.0,38.5,stable,3.6
17907,content_5ec29ae79c60,18184.40,POOR_POSITION,60500.0,49.8,up,73.0
28282,content_454cc6654c6e,18176.32,DECLINING_TRAFFIC,60500.0,44.9,down,-52.8
6972,content_bf67a444faef,18163.20,DECLINING_TRAFFIC,60500.0,45.5,down,-20.0
18701,content_deb54e9e19cd,18156.20,POOR_POSITION,60500.0,41.7,stable,-2.5
24695,content_dd882c4152ac,17982.00,POOR_POSITION,70.0,75.5,up,44900.0
16005,content_83e3da1394ac,14938.34,POOR_POSITION,49500.0,65.5,up,218.1
13502,content_f76ccf7a7834,14927.26,DECLINING_TRAFFIC,49500.0,9.5,down,-37.9
22788,content_ee4630879d03,14892.32,DECLINING_TRAFFIC,49500.0,25.2,down,-29.3
8055,content_cd6760921db8,14878.05,DECLINING_TRAFFIC,49500.0,47.3,down,-65.0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


Weak Picks:

Some pages that were ranked very high were chosen because one factor was very strong while the other factors were not as strong. These pages need to be looked at before taking any action.

Leakage Check:

The rule only uses information that's available when a decision is made:

1.Search volume

2.Average position

3.Trend direction

4.Trend percentage

5.Days since last update

No information, from the future or columns that were created from labels was used. No leakage was found.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.